# TabICL L4 (Colab) — exp_071 raw full 5-fold (범주형 자동인코딩)
**런타임 → 런타임 유형 변경 → L4 GPU** 선택. 좌측 🔑 Secrets에 **KAGGLE_USERNAME · KAGGLE_KEY** 등록(노트북 액세스 ON) 후 위에서부터 실행.
OOF는 마지막 셀에서 다운로드 → `/teamspace .../experiments/oof/`에 복사. 실행법 SSOT=docs/wiki/colab_jobs.md

In [ ]:
# 1) 설치 (tabicl + 우리 src 의존)
!pip install -q tabicl hydra-core omegaconf python-dotenv scikit-learn pandas

import torch

print('CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))  # L4 확인

In [ ]:
# 2) 인증 — Colab Secrets(KAGGLE_USERNAME/KAGGLE_KEY/WANDB_API_KEY). kaggle.json 업로드 불요
from google.colab import userdata
import os

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

print('인증 OK (Colab Secrets: Kaggle + W&B online)')

In [ ]:
# 3) 데이터(대회) + 코드(f1-pit-src) 다운로드
!kaggle competitions download -c playground-series-s6e5 -p /content/comp -q && cd /content/comp && unzip -oq '*.zip'
!kaggle datasets download -d buzziru/f1-pit-src -p /content/srcd -q && cd /content/srcd && unzip -oq '*.zip'
!ls /content/comp && echo '---' && ls /content/srcd

In [ ]:
# 4) src import + 경로 override (Colab 경로로)
import sys
sys.path.insert(0, '/content/srcd')

from pathlib import Path
from src import config

config.TRAIN_PATH = Path('/content/comp/train.csv')
config.TEST_PATH = Path('/content/comp/test.csv')
config.SAMPLE_SUBMISSION_PATH = Path('/content/comp/sample_submission.csv')

out = Path('/content/out')
config.OOF_DIR = out / 'oof'
config.SUBMISSION_DIR = out / 'submissions'
config.LOG_DIR = out / 'logs'
for d in [config.OOF_DIR, config.SUBMISSION_DIR, config.LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

from src.train_tabicl import run

print('import OK')

In [ ]:
# 4.5) 소규모 fast-fail (10k — L4 24GB 메모리·API 검증)
import pandas as pd
import time
from tabicl import TabICLClassifier

_tr = pd.read_csv(config.TRAIN_PATH).sample(10000, random_state=42)
_y = _tr['PitNextLap']
_X = _tr.drop(columns=['id', 'PitNextLap'])
# 범주형은 변환 없이 그대로 — TabICL 자동 인코딩(cat.codes ordinal 금지)

t0 = time.time()
m = TabICLClassifier(device='cuda', n_estimators=8, batch_size=2, offload_mode='auto', random_state=42)
m.fit(_X, _y)

print(f'10k OK {time.time()-t0:.0f}s | L4면 full 진행')

In [ ]:
# 5) full 5-fold run (base raw, augment False)
from omegaconf import OmegaConf
import time

CONF = Path('/content/srcd/conf')
cfg = OmegaConf.create({
    'exp_id': 'exp_071_tabicl_raw_full',
    'notes': 'TabICL L4 full 5-fold: base + 범주형 자동인코딩(cat.codes 제거) → 5-member 스택 게이트',
    'use_wandb': True,
    'max_folds': None,
    'model': OmegaConf.load(CONF / 'model' / 'tabicl.yaml'),
    'features': OmegaConf.load(CONF / 'features' / 'base.yaml'),
    'augment': {'enabled': False, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))

t0 = time.time()
result = run(cfg)

print(result, f'{time.time()-t0:.0f}s')
print('cv_mean =', result.get('cv_mean'), '| fold_scores =', result.get('fold_scores'))

In [ ]:
# 6) 로그·OOF·submission 다운로드 → /teamspace .../experiments/{logs,oof,submissions}/ 에 복사
# ⚠️ OOF·submission 파일명이 동일(exp_071_tabicl_raw_full.csv) → 그대로 download 시 브라우저가 (1) 중복처리.
#    구분 접미사(_log/_oof/_sub)로 임시 복사 후 download (받은 파일명 = 표준 이름에서 접미사만 떼면 됨).
from google.colab import files
import shutil

eid = 'exp_071_tabicl_raw_full'
targets = [
    (config.LOG_DIR / f'{eid}.json', f'{eid}_log.json'),
    (config.OOF_DIR / f'{eid}.csv', f'{eid}_oof.csv'),
    (config.SUBMISSION_DIR / f'{eid}.csv', f'{eid}_sub.csv'),
]

for src, name in targets:
    dst = f'/content/{name}'
    shutil.copy(src, dst)
    files.download(dst)